# 10 · Fit each guide's effect on each gene

A negative binomial generalised linear model per response gene, with the
knockout guides as covariates. Counts are modelled directly rather than
log-normalised values, so the mean-variance relationship of the data is
respected.

**Reads** `par_save_filename_7`. **Writes** coefficient tables into
`par_guide_lm_dir`, which notebook 12 reads.

The model is `MASS::glm.nb`, called through `rpy2`. Guides are taken in blocks
of `par_guide_block_size` and genes in blocks of `par_gene_block_size`; each
gene block is written as it completes, so an interrupted run resumes where it
stopped.

This is the longest-running notebook in the series. Set
`par_guide_block_start` from outside with `papermill` to fit one guide block per
process and run them in parallel.

:::{note}
The fits below take days to run across all guides, so their results are
provided rather than regenerated. `TextFiles/GuideSelect_BadKOGuides.csv` and
`TextFiles/GuideSelect_GoodGuides.csv` hold the guide selection the published
analysis used, and notebook 13 reads the first of them directly.

This notebook records how that selection was made. It writes its own results to
separate files under `outputs/` and leaves the provided lists untouched.
:::


## Setup

In [ ]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.conversion import localconverter

FIT_DIR = par_guide_lm_dir + "/NegativeBinomial"
Path(FIT_DIR).mkdir(parents=True, exist_ok=True)

## The R side

Defined once as a function rather than issued as cell magics, so it can be
called from a loop and the failure of a single gene does not stop the block.
A gene whose model does not converge is skipped and reported.

In [ ]:
ro.r('''
library("MASS")
library("broom")

fit_nb_block <- function(guide_df, expr_mat, model_formula, resp_names) {
    out <- data.frame()
    for (j in seq_len(ncol(expr_mat))) {
        tryCatch({
            guide_df[["y"]] <- expr_mat[, j]
            fit <- glm.nb(as.formula(model_formula), data = guide_df)
            tidied <- data.frame(tidy(fit))
            tidied$respGene <- resp_names[j]
            out <- rbind(out, tidied)
        }, error = function(e) {
            message(paste("  skipped", resp_names[j], ":", conditionMessage(e)))
        })
    }
    out
}
''')
print("fit_nb_block defined")

In [ ]:
adata = sc.read(par_save_filename_7)
print(f"input: {adata.shape[0]} cells x {adata.shape[1]} genes")

ko_guides = list(adata.uns["feature_KO_barcode_names_filtered"])
control_guides = [
    g for g in adata.uns["feature_barcode_names"]
    if g.startswith((par_not_target_control_prefix, par_nongene_site_control_prefix))
]
adata.obs["ControlGuidesAll"] = adata.obs[control_guides].sum(axis=1)

# raw counts, in the gene order of the filtered object
raw_X = adata.raw.X.toarray() if hasattr(adata.raw.X, "toarray") else adata.raw.X
expression = pd.DataFrame(raw_X, index=adata.obs_names,
                          columns=adata.raw.var_names)[adata.var_names]

covariates = adata.obs[["n_genes", "mt_frac", "leiden"]]
guide_matrix = adata.obs[ko_guides].join(covariates)
response_genes = list(adata.var_names)
print(f"knockout guides: {len(ko_guides)}, response genes: {len(response_genes)}")

## Fit

Each guide block is fitted against its own cells: those carrying one of the
block's guides, plus a fixed panel of `par_nb_control_cells` control cells as
the reference population.

In [ ]:
starts = (list(range(0, len(ko_guides), par_guide_block_size))
          if par_guide_block_start is None else [par_guide_block_start])

for block_start in starts:
    block_stop = min(block_start + par_guide_block_size, len(ko_guides))
    block_guides = ko_guides[block_start:block_stop]
    selected = block_guides + ["n_genes", "mt_frac", "leiden"]
    formula = "y~" + "+".join(selected)
    print(f"\nguide block {block_start}-{block_stop}")

    is_control = adata.obs["ControlGuidesAll"] == 1
    control_expr = expression.loc[is_control].iloc[:par_nb_control_cells]
    control_guide = guide_matrix.loc[is_control, selected].iloc[:par_nb_control_cells]

    in_block = adata.obs[block_guides].sum(axis=1) > 0
    block_expr = expression.loc[in_block]
    block_guide = guide_matrix.loc[in_block, selected]

    all_expr = np.concatenate((control_expr.to_numpy(), block_expr.to_numpy()))
    all_guide = pd.concat([control_guide, block_guide])
    print(f"  cells: {all_guide.shape[0]} ({control_guide.shape[0]} control)")

    for gene_start in range(0, len(response_genes), par_gene_block_size):
        gene_stop = min(gene_start + par_gene_block_size, len(response_genes))
        out = f"{FIT_DIR}/coefs_{block_start}_{block_stop}_{gene_start}_{gene_stop}.csv"
        if Path(out).exists():
            continue

        names = response_genes[gene_start:gene_stop]
        sub_expr = all_expr[:, gene_start:gene_stop]

        with localconverter(ro.default_converter + pandas2ri.converter + numpy2ri.converter):
            fitted = ro.globalenv["fit_nb_block"](
                all_guide, sub_expr, formula, ro.StrVector(names)
            )
            fitted = ro.conversion.rpy2py(fitted)

        fitted.to_csv(out, index=False)
        print(f"  genes {gene_start}-{gene_stop}: {len(fitted)} rows")

print("\nall blocks fitted")